In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
#fix seeds so runs are responsible 
torch.manual_seed(0)
np.random.seed(0)

In [ ]:
data = np.load(r'C:\Users\kteja\Downloads\learning-physical-dynamics-across-resolutions-group-12\learning-physical-dynamics-across-resolutions-group-12\trainv')
X = data['field'][:, None, :].astype(np.float32)
Y = data['target'][:, None, :].astype(np.float32)
print("Data:", X.shape, Y.shape)

In [ ]:
# Trajectory-level split, NOT row-level.
# Row-level would leak: rows t*5..t*5+4 belong to the same trajectory, so
# neighbouring time steps of the same physics would land in both train and val.
# We hold out 10 whole trajectories (50 examples) for validation.
n_traj = 48
perm = np.random.permutation(n_traj)
val_traj = perm[:10]
train_traj = perm[10:]
val_idx = np.concatenate([np.arange(t*5, t*5+5) for t in val_traj])
train_idx = np.concatenate([np.arange(t*5, t*5+5) for t in train_traj])

In [ ]:
X_train = torch.tensor(X[train_idx]); Y_train = torch.tensor(Y[train_idx])
X_val   = torch.tensor(X[val_idx]);   Y_val   = torch.tensor(Y[val_idx])

Grid-locked CNN baseline: kernel defined in grid points, not physical distance.
Effective receptive field in physical space shrinks as the grid is refined, so the same weights cannot resolve the same features at higher resolutions.

Architecture: 4 layers, kernel 3, hidden 64, circular padding, residual.
Chosen by ablation over depth {2,4,6,8}, kernel {3,5,7}, width {32,64,128};
selected the smallest config within noise of the best validation loss, refer to ablation.py

In [ ]:
class GridLockedCNN(nn.Module):

    def __init__(self, hidden=64, kernel_size=3, num_layers=4):
        super().__init__()
        #Circular padding is chosen since theBurgers domain is periodic. Matches CNN baseline convention in Li et al. 2021 and FNO tutorial.
        layers = [nn.Conv1d(1, hidden, kernel_size,
                            padding=kernel_size // 2,
                            padding_mode='circular'), nn.ReLU()]
        for _ in range(num_layers - 1):
            layers += [nn.Conv1d(hidden, hidden, kernel_size,
                                 padding=kernel_size // 2,
                                 padding_mode='circular'), nn.ReLU()]
        layers.append(nn.Conv1d(hidden, 1, kernel_size,
                                padding=kernel_size // 2,
                                padding_mode='circular'))
        self.net = nn.Sequential(*layers)
# Residual form: predict the update, not the full state.
    # Matches the neural-operator layer structure (Kovachki et al. 2023, eq. 6) and stabilises training on small data.
    def forward(self, u):
        return u + self.net(u)

In [ ]:
def relative_l2(pred, target):
    num = torch.norm(pred - target, dim=(1, 2))
    den = torch.norm(target, dim=(1, 2)) + 1e-8
    return (num / den).mean()

In [ ]:
model = GridLockedCNN(hidden=64, kernel_size=3, num_layers=4)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=32, shuffle=True)
# Callbacks:
#  early stopping (patience 40): small dataset leads to overfitting risk hence stop  when val loss has not improved for 40 epochs.
#  reduce LR on plateau (patience 20): Adam plateaus on small data; halving the LR escapes shallow minima.
#   best-checkpoint saving: preserve the model with the lowest val loss, not the last-epoch model
best_val = float('inf'); wait = 0; lr_wait = 0
train_hist, val_hist = [], []

In [ ]:
for epoch in range(600):
    model.train()
    total = 0.0
    for xb, yb in loader:
        loss = relative_l2(model(xb), yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * len(xb)
    tl = total / len(X_train)

    model.eval()
    with torch.no_grad():
        vl = relative_l2(model(X_val), Y_val).item()

    train_hist.append(tl); val_hist.append(vl)
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}  train={tl:.6f}  val={vl:.6f}")

    if vl < best_val - 1e-6:
        best_val = vl; wait = 0; lr_wait = 0
        torch.save(model.state_dict(), 'cnn_128.pt')
    else:
        wait += 1; lr_wait += 1
#Small dataset means risk of overfitting. Early stop protects against it
    if wait >= 40:
        print(f"Early stop at {epoch}. Best val = {best_val:.6f}")
        break
    if lr_wait >= 20:
        for g in optimizer.param_groups:
            g['lr'] *= 0.5
        print(f"  LR -> {optimizer.param_groups[0]['lr']:.1e}")
        lr_wait = 0
#Preserves best-val model instead of last-epoch model
np.savez('training_history.npz', train=train_hist, val=val_hist, best=best_val)
print(f"Best val = {best_val:.6f}")